# Мини-проект по прогнозированию стоимости домов

В рамках работы используются табличные данные о недвижимости. Для решения задачи рассматриваются несколько регрессионных моделей: линейная регрессия, Ridge, Lasso и Random Forest Regressor. Особое внимание уделяется обработке пропусков, кодированию категориальных признаков, логарифмической трансформации целевой переменной и сравнению качества моделей.

## Linear Regression

В качестве базовой модели используется линейная регрессия. Эта модель служит отправной точкой для дальнейшего сравнения с регуляризованными линейными методами и ансамблевыми алгоритмами.

Линейная регрессия рассматривается без отдельного подбора гиперпараметров, так как в рамках данной задачи она используется прежде всего как baseline-модель.

## Ridge Regression

Для улучшения базовой линейной модели используется Ridge Regression — линейная регрессия с L2-регуляризацией.

Регуляризация позволяет уменьшить переобучение и сделать модель более устойчивой при наличии большого числа признаков, в том числе после one-hot кодирования категориальных переменных.

## Lasso Regression

Следующая модель — Lasso Regression, использующая L1-регуляризацию.

Особенность Lasso заключается в том, что модель может занулять часть коэффициентов, тем самым выполняя встроенный отбор признаков. Это делает модель полезной для сравнения с Ridge и обычной линейной регрессией.

## Random Forest Regressor

Для учета нелинейных зависимостей в данных используется ансамблевая модель `RandomForestRegressor`.

Случайный лес состоит из множества деревьев решений, обученных на разных подвыборках данных, а итоговый прогноз формируется как усреднение предсказаний всех деревьев. Такой подход обычно дает более устойчивое качество по сравнению с одной линейной моделью.

## Импорты библиотек и отключение предупреждений

In [37]:

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

## Работа с датафреймом
Создание тестовых и отложенных выборок, логарифмическая трансформация данных, заполнение пропусков в данных.

In [ ]:
train = pd.read_csv('../data/train.csv', sep=',')
y = np.log1p(train['SalePrice'])
X = train.drop(['SalePrice'], axis=1)

In [39]:
nums = X.select_dtypes(include=['int64', 'float64']).columns
categories = X.select_dtypes(include=['object']).columns

nums_transform = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categories_transform = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', nums_transform, nums),
    ('cat', categories_transform, categories)
])

In [40]:
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=17)

In [41]:
cv = KFold(n_splits=5, shuffle=True, random_state=17)

## Линейная регрессия

In [42]:
linreg = Pipeline([('preprocessor', preprocessor), 
                   ('regressor', LinearRegression())])


In [43]:
linreg_result = cross_val_score(linreg, X_train,y_train, n_jobs=-1, scoring='neg_root_mean_squared_error',cv=cv)
print('CV RMSE (log пространство): ', np.expm1(-linreg_result.mean()))

CV RMSE (log пространство):  0.17373299554960184


In [44]:
linreg.fit(X_train, y_train)

linreg_pred = np.expm1(linreg.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), linreg_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), linreg_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 15442.372876748066
Holdout RMSE: 22270.21329785158


## Ridge

In [45]:
ridge = Pipeline([('preprocessor', preprocessor), 
                   ('regressor', Ridge())])

In [46]:
ridge_result = cross_val_score(ridge, X_train,y_train, n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
print('CV RMSE (log пространство): ', np.expm1(-ridge_result.mean()))

CV RMSE (log пространство):  0.1562949637185474


In [47]:
ridge.fit(X_train, y_train)

ridge_pred = np.expm1(ridge.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), ridge_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), ridge_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 15216.33936229802
Holdout RMSE: 21913.028793718364


## Подбор гиперпараметров для Ridge Regression

Для Ridge Regression подбирается параметр `alpha`, который определяет силу регуляризации.

Малые значения `alpha` соответствуют слабой регуляризации, а большие — более сильной. Подбор выполняется с помощью `GridSearchCV`, а качество модели оценивается по метрике RMSE в логарифмической шкале целевой переменной.

In [48]:
ridge_params = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

In [49]:
ridge_pipe = Pipeline([('preprocessor', preprocessor), 
                   ('regressor', Ridge())])

In [50]:
best_ridge = GridSearchCV(estimator=ridge_pipe, param_grid=ridge_params,n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
best_ridge.fit(X_train,y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=17, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea...
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                                       ('regressor', Ridge())]),
             n_jobs=-1,
             param_grid={'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0,
                                              100.0, 1000.0]},
             scoring='neg_root_mean_squared_error')

In [51]:
print('CV RMSE (log пространство): ', np.expm1(-best_ridge.best_score_))

CV RMSE (log пространство):  0.15094282967336903


In [52]:
best_ridge_pred = np.expm1(best_ridge.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), best_ridge_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), best_ridge_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 14836.047605498283
Holdout RMSE: 21489.69704658593


## Lasso

In [53]:
lasso1 = Pipeline([('preprocessor', preprocessor), 
                   ('regressor', Lasso(max_iter=10000, random_state=17))])

In [54]:
lasso_result = cross_val_score(lasso1, X_train,y_train, n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
print('CV RMSE (log пространство): ', np.expm1(-lasso_result.mean()))

CV RMSE (log пространство):  0.4898480521683761


In [55]:
lasso1.fit(X_train, y_train)

lasso_pred = np.expm1(lasso1.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), lasso_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), lasso_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 56158.34840905439
Holdout RMSE: 79334.47160657092


## Подбор гиперпараметров для Lasso Regression

Для Lasso Regression подбирается параметр `alpha`, отвечающий за силу регуляризации.

Так как Lasso чувствительна к величине регуляризации и может слишком агрессивно занулять коэффициенты, подбор `alpha` особенно важен для получения устойчивого результата.

In [56]:
lasso_params = {
    'regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]
}

In [57]:
lasso_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=10000, random_state=17))
])

In [58]:
best_lasso = GridSearchCV(lasso_pipe, lasso_params, n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
best_lasso.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=17, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea...
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                                       ('regressor',
                                        Lasso(max_iter=10000,
                                              random_state=17))]),
             n_jobs=-1,
             param_grid={'regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]},
             scoring='neg_root_mean_squared_error')

In [59]:
print('CV RMSE (log пространство): ', np.expm1(-best_lasso.best_score_))

CV RMSE (log пространство):  0.15142213465384258


In [60]:
best_lasso_pred = np.expm1(best_lasso.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), best_lasso_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), best_lasso_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 14595.224571673632
Holdout RMSE: 21105.181866357565


## Random Forest Regression

In [61]:
forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=17, n_jobs=-1))
])

In [62]:
forest_result = cross_val_score(forest, X_train,y_train, n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
print('CV RMSE (log пространство): ', np.expm1(-forest_result.mean()))

CV RMSE (log пространство):  0.15967924719714077


In [63]:
forest.fit(X_train, y_train)

forest_pred = np.expm1(forest.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), forest_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), forest_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 17936.35295999644
Holdout RMSE: 28382.921630262997


## Подбор гиперпараметров для Random Forest Regressor

Для случайного леса подбираются параметры, влияющие на сложность и устойчивость ансамбля:

- `n_estimators` — количество деревьев;
- `max_depth` — максимальная глубина деревьев;
- `min_samples_split` — минимальное число объектов для разбиения узла;
- `min_samples_leaf` — минимальное число объектов в листе;
- `max_features` — число признаков, рассматриваемых при поиске лучшего разбиения.

Подбор выполняется с помощью `GridSearchCV` по метрике RMSE в логарифмической шкале целевой переменной.

In [64]:
forest_params = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2']
}

In [65]:
forest_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=17))
])

In [66]:
best_forest = GridSearchCV(forest_pipe, forest_params, n_jobs=-1, scoring='neg_root_mean_squared_error', cv=cv)
best_forest.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=17, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea...
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=17))]),
             n_jobs=-1,
             param_grid={'regressor__max_depth': [None, 10, 20],
                         'regressor__max_features': ['sqrt', 'log2'],
                         'regressor__min_samples_leaf': [1, 2, 4],
                         'regressor__min_samples_split': [2, 5, 10],
                         'regressor__n_estimators': [100, 200, 300]},
             scoring='neg_root_mean_squared_error')

In [67]:
print('CV RMSE (log пространство): ', np.expm1(-best_forest.best_score_))

CV RMSE (log пространство):  0.15800209609585458


In [68]:
best_forest_pred = np.expm1(best_forest.predict(X_holdout))

mae = mean_absolute_error(np.expm1(y_holdout), best_forest_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_holdout), best_forest_pred))

print('Holdout MAE:', mae)
print('Holdout RMSE:', rmse)

Holdout MAE: 18306.55056344617
Holdout RMSE: 30437.25753981345


In [69]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Ridge',
        'Ridge (params)',
        'Lasso',
        'Lasso (params)',
        'Random Forest',
        'Random Forest (params)'
    ],
    'CV RMSE (log scale)': [
        -linreg_result.mean(),
        -ridge_result.mean(),
        -best_ridge.best_score_,
        -lasso_result.mean(),
        -best_lasso.best_score_,
        -forest_result.mean(),
        -best_forest.best_score_
    ],
    'Holdout MAE': [
        mean_absolute_error(np.expm1(y_holdout), np.expm1(linreg.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(ridge.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(best_ridge.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(lasso1.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(best_lasso.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(forest.predict(X_holdout))),
        mean_absolute_error(np.expm1(y_holdout), np.expm1(best_forest.predict(X_holdout)))
    ],
    'Holdout RMSE': [
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(linreg.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(ridge.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(best_ridge.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(lasso1.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(best_lasso.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(forest.predict(X_holdout)))),
        np.sqrt(mean_squared_error(np.expm1(y_holdout), np.expm1(best_forest.predict(X_holdout))))
    ]
})

results = results.sort_values(by='Holdout RMSE').reset_index(drop=True)
results

,Model,CV RMSE (log scale),Holdout MAE,Holdout RMSE
0,Lasso (params),0.140998,14595.224572,21105.181866
1,Ridge (params),0.140581,14836.047605,21489.697047
2,Ridge,0.145221,15216.339362,21913.028794
3,Linear Regression,0.160189,15442.372877,22270.213298
4,Random Forest,0.148143,17936.352960,28382.921630
5,Random Forest (params),0.146696,18306.550563,30437.257540
6,Lasso,0.398674,56158.348409,79334.471607


## Итоговые выводы

В ходе работы были рассмотрены несколько регрессионных моделей для задачи прогнозирования стоимости домов: Linear Regression, Ridge Regression, Lasso Regression и Random Forest Regressor.

Для всех моделей использовалась единая схема предобработки данных, включающая заполнение пропусков, масштабирование числовых признаков и one-hot кодирование категориальных переменных. Дополнительно к целевой переменной `SalePrice` была применена логарифмическая трансформация, что позволило уменьшить асимметрию распределения и сделать обучение моделей более устойчивым.

По итогам сравнения моделей на кросс-валидации и на отложенной выборке можно определить наиболее удачный подход для данной задачи. Линейная регрессия выступила как базовая модель, Ridge и Lasso показали влияние регуляризации на качество прогноза, а Random Forest позволил проверить эффективность нелинейного ансамблевого метода.

Таким образом, проект показал, как выбор модели, настройка гиперпараметров и корректная предобработка данных влияют на итоговое качество прогноза стоимости недвижимости.